# Auto-encodeurs simples

## Installation et import de PyTorch Lightning et des autres librairies nécessaires

In [ ]:
!pip install -q lightning

In [ ]:
from collections import OrderedDict

import lightning
import matplotlib.pyplot as plt
import scipy.interpolate
import torch
import torchvision
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

## Chargement de MNIST

Nous allons utiliser un prétraîtement légèrement différent des autres fois : étant donné que nous voulons pouvoir prédire les valeurs données en entrée en sortie (principe de l'auto-encodage), nous allons simplement projeter ces valeurs dans $[0, 1]$ au lieu de $[0, 255]$. Notez qu'habituellement nous ne faisons pas ça : nous normalisons en centrant sur zéro et en divisant par l'écart-type.

In [ ]:
train_data = torchvision.datasets.MNIST("data", train=True, download=True)
test_data = torchvision.datasets.MNIST("data", train=False, download=True)
nb_classes = 10
input_dim = 28 * 28
X_train = train_data.data.reshape(-1, input_dim).float() / 255.0
y_train = train_data.targets
X_test = test_data.data.reshape(-1, input_dim).float() / 255.0
y_test = test_data.targets

In [ ]:
X_train.shape

## Création de l'autoencodeur

Vous devriez être capable de créer le modèle d'autoencoder de base par vous-même.

Attention aux choix des fonctions d'activations et loss !


In [ ]:
class AutoEncoder(lightning.LightningModule):
  """Lightning wrapper: the input is also the target to reconstruct."""

  def __init__(self,
               model: nn.Module,
               loss: nn.Module,
               learning_rate: float = 1e-3) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model", "loss"])
    self.model = model
    self.loss = loss
    # Une entrée d'exemple permet à Lightning d'afficher la forme des tenseurs
    # d'entrée et de sortie de chaque couche dans le résumé du modèle
    self.example_input_array = torch.zeros(1, input_dim)

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.model(images)

  def _step(self, batch: tuple[torch.Tensor], stage: str) -> torch.Tensor:
    images, = batch
    loss = self.loss(self(images), images)
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True,
             prog_bar=True)
    return loss

  def training_step(self, batch: tuple[torch.Tensor],
                    batch_index: int) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(self, batch: tuple[torch.Tensor],
                      batch_index: int) -> torch.Tensor:
    return self._step(batch, "val")

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(),
                            lr=self.hparams.learning_rate)


@torch.no_grad()
def predict(model: nn.Module,
            X: torch.Tensor,
            batch_size: int = 256) -> torch.Tensor:
  """Apply a network to every row of X, batch by batch."""
  device = next(model.parameters()).device
  model.eval()
  return torch.cat([model(batch.to(device)).cpu()
                    for batch in X.split(batch_size)])

In [ ]:
# Votre code ici
encoding_dim = 20
encoder = nn.Sequential()
decoder = nn.Sequential()
autoencoder = nn.Sequential()

### Solution

In [ ]:
encoding_dim = 20
encoder = nn.Sequential(
    nn.Linear(input_dim, encoding_dim),
    nn.ReLU())

decoder = nn.Sequential(
    nn.Linear(encoding_dim, input_dim),
    nn.Sigmoid())

autoencoder = nn.Sequential(OrderedDict(encoder=encoder, decoder=decoder))

model = AutoEncoder(autoencoder, loss=nn.MSELoss())
print(ModelSummary(model, max_depth=2))

In [ ]:
encoding_dim = 20


def dense(in_features: int,
          out_features: int,
          activation: type[nn.Module] | None = nn.ReLU) -> nn.Module:
  layer = nn.Linear(in_features, out_features)
  nn.init.orthogonal_(layer.weight)
  if activation is None:
    return layer
  return nn.Sequential(layer, activation())


encoder = nn.Sequential(
    dense(input_dim, 150),
    dense(150, 150),
    dense(150, 150),
    dense(150, 150),
    dense(150, 50),
    dense(50, encoding_dim, activation=None))

decoder = nn.Sequential(
    dense(encoding_dim, 50),
    dense(50, 150),
    dense(150, 150),
    dense(150, 150),
    dense(150, 150),
    dense(150, input_dim, activation=nn.Sigmoid))

autoencoder = nn.Sequential(OrderedDict(encoder=encoder, decoder=decoder))

model = AutoEncoder(autoencoder, loss=nn.MSELoss())
print(ModelSummary(model, max_depth=-1))

## Apprentissage

*Écrivez les lignes correspondant à l'apprentissage de votre autoencodeur :*

- *50 itérations devraient suffire*
- *Utilisez un batch de 256*

In [ ]:
# Votre code ici

### Solution

In [ ]:
batch_size = 256

# 20 % des données d'entraînement sont mises de côté pour la validation
train_dataset, val_dataset = random_split(TensorDataset(X_train), [0.8, 0.2])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

trainer = lightning.Trainer(max_epochs=50,
                            accelerator="auto",
                            devices=1,
                            logger=CSVLogger("logs", name="autoencoder"),
                            enable_checkpointing=False)
trainer.fit(model, train_loader, val_loader)

## Prédiction sur du bruit blanc

Pour une première utilisation du décodeur, on peut regarder ce qu'il prédit sur du bruit blanc en entrée.

*Parfois, l'image générée est quasiment constante malgré le bruit aléatoire donné en entrée. À quoi cela est-il dû ?*

In [ ]:
white_noise = torch.rand(1, encoding_dim)
plt.imshow(predict(decoder, white_noise).reshape(28, 28), cmap="gray_r")
plt.show()

## Encodage des données de test

On peut aussi, grâce à l'encodeur récupéré, encoder nos données de test vers l'espace de dimension `encoding_dim` appris.

In [ ]:
codes = predict(encoder, X_test)

In [ ]:
codes.shape

## Calcul des centroïdes de chaque chiffre dans l'espace de codage

Un autre point intéressant est de regarder en moyenne où atterrissent les exemples d'un label donné dans l'espace de codage.

In [ ]:
means = torch.stack([codes[y_test == i].mean(dim=0)
                     for i in range(nb_classes)])
stds = torch.stack([codes[y_test == i].std(dim=0)
                    for i in range(nb_classes)])

# Étude rapide des codages en test
for i in range(10):
  dimension_stats = [f"{mean:5.2f}±{std:.2f}"
                     for mean, std in zip(means[i], stds[i])]
  print(f"Classe {i} {', '.join(dimension_stats)}")

On peut à partir de là vérifier que chaque centroïde est bien décodé par quelque chose de vraisemblable par le décodeur.

In [ ]:
f, ax = plt.subplots(1, nb_classes, figsize=(1.4 * nb_classes, 2))

centroid_images = predict(decoder, means).reshape(-1, 28, 28)

for i, centroid_image in enumerate(centroid_images):
  ax[i].imshow(centroid_image, cmap="gray_r")
  ax[i].axis("off")
plt.show()

## Parcours de l'espace latent entre deux centroïdes

Maintenant que l'on sait où se trouvent les centroïdes pour un chiffre donné dans l'espace latent, on peut parcourir l'espace entre deux centroïdes pour mieux comprendre comment l'espace latent est structuré.

In [ ]:
def latent_walk(start: int, end: int, n: int = 15):
  interpolator = scipy.interpolate.interp1d([0, n - 1],
                                            means[[start, end], :].numpy(),
                                            axis=0)
  interpolated_codes = torch.tensor(interpolator(range(n)),
                                    dtype=torch.float32)
  interpolated_images = predict(decoder,
                                interpolated_codes).reshape(-1, 28, 28)

  f, ax = plt.subplots(1, n, figsize=(n * 1.4, 2))
  for i, interpolated_image in enumerate(interpolated_images):
    ax[i].imshow(interpolated_image, cmap="gray_r")
    ax[i].axis("off")
  plt.show()


latent_walk(7, 6)

## Auto-encodage de toute la base de test

Auto-encodez les images de train et de test et stockez les images obtenues dans les variables `X_train_pred` et `X_test_pred`

In [ ]:
# Votre code ici
X_train_pred = X_train
X_test_pred = X_test

### Solution

In [ ]:
X_train_pred = predict(autoencoder, X_train)
X_test_pred = predict(autoencoder, X_test)

## Évaluation visuelle de la performance

In [ ]:
n = 15  # Nombre de chiffres que nous allons afficher

random_indexes = torch.randperm(X_test_pred.shape[0])[:n]


def display_samples(indices: torch.Tensor,
                    X: torch.Tensor,
                    X_pred: torch.Tensor,
                    y: torch.Tensor
                   ) -> None:
  _, ax = plt.subplots(2, len(indices), figsize=(len(indices) * 1.4, 4))
  for i, index in enumerate(indices):
    # L'original en haut
    ax[0, i].set_title(int(y[index]))
    ax[0, i].imshow(X[index].reshape(28, 28), cmap="gray_r")
    ax[0, i].axis("off")

    # La reconstruction en bas
    ax[1, i].imshow(X_pred[index].reshape(28, 28), cmap="gray_r")
    ax[1, i].axis("off")
  plt.show()


display_samples(random_indexes, X_test, X_test_pred, y_test)

## Détection d'anomalies

Il est possible de détecter des anomalies dans les données en utilisant l'erreur de reconstruction : une grande erreur induit que l'exemple est hors de la distribution normale des données.

In [ ]:
def reconstruction_errors(X: torch.Tensor,
                          X_pred: torch.Tensor) -> torch.Tensor:
  """Mean squared error of each reconstruction."""
  return ((X_pred - X) ** 2).mean(dim=-1)


test_anomalies = reconstruction_errors(X_test,
                                       X_test_pred).argsort(descending=True)
train_anomalies = reconstruction_errors(X_train,
                                        X_train_pred).argsort(descending=True)

n = 20

print("Pires reconstructions sur le train")
display_samples(train_anomalies[:n], X_train, X_train_pred, y_train)

print("Pires reconstructions sur le test")
display_samples(test_anomalies[:n], X_test, X_test_pred, y_test)

print("Meilleures reconstructions sur le train")
display_samples(train_anomalies[-n:], X_train, X_train_pred, y_train)

print("Meilleures reconstructions sur le test")
display_samples(test_anomalies[-n:], X_test, X_test_pred, y_test)

## À vous de jouer !

Trouvez une valeur idéale pour la taille de l'encodage ainsi qu'un modèle adapté afin que votre autoencodeur ait une perte de compression acceptable.

Quelle est votre taux de compression ?